# Wiktionary Phrasal Verb Integration Playground

This notebook inspects the newly integrated Wiktionary phrasal-verb supplement and compares lexicon coverage before vs after integration.

## What Was Added
- Source: `Category:English phrasal verbs` from Wiktionary MediaWiki API.
- Local file: `data/lexicons/phrasal_verbs_wiktionary_category.json`.
- Merge behavior in app: add entries not already present in WeCan/Semigradsky, with default metadata (`separable=False`, empty definition/examples).
- Goal: improve coverage of common phrasal verbs missing from previous sources.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 120)

In [2]:
cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'app').exists() else cwd.parent
LEX = ROOT / 'data' / 'lexicons'

idioms = json.loads((LEX / 'idioms_mcgrawhill.json').read_text(encoding='utf-8'))['dictionary']
wecan = json.loads((LEX / 'phrasal_verbs_wecan.json').read_text(encoding='utf-8'))
semi = json.loads((LEX / 'phrasal_verbs_semigradsky.json').read_text(encoding='utf-8'))
wiki_payload = json.loads((LEX / 'phrasal_verbs_wiktionary_category.json').read_text(encoding='utf-8'))
wiki_phrases = wiki_payload['phrases']

print('idioms:', len(idioms))
print('wecan:', len(wecan))
print('semigradsky:', len(semi))
print('wiktionary category:', len(wiki_phrases))

idioms: 21866
wecan: 3350
semigradsky: 124
wiktionary category: 5068


In [3]:
def parse_semigradsky(verb_field: str) -> str:
    clean = verb_field.replace('+', '').replace('*', '').strip()
    return ' '.join(clean.split())


base_set = set(wecan.keys())
base_set.update(parse_semigradsky(e.get('verb', '')) for e in semi)
base_set = {x for x in base_set if x and len(x.split()) >= 2}

wiki_set = {x for x in wiki_phrases if x and len(x.split()) >= 2}

merged_set = set(base_set)
merged_set.update(wiki_set)

stats = pd.DataFrame([
    {'metric': 'base_wecan_plus_semigradsky', 'count': len(base_set)},
    {'metric': 'wiktionary_category', 'count': len(wiki_set)},
    {'metric': 'overlap_base_and_wiktionary', 'count': len(base_set & wiki_set)},
    {'metric': 'added_from_wiktionary_only', 'count': len(wiki_set - base_set)},
    {'metric': 'final_merged_total', 'count': len(merged_set)},
])
stats

,metric,count
0,base_wecan_plus_semigradsky,3354
1,wiktionary_category,5068
2,overlap_base_and_wiktionary,2436
3,added_from_wiktionary_only,2632
4,final_merged_total,5986


In [4]:
sample_new = sorted(wiki_set - base_set)[:40]
pd.DataFrame({'new_entries_from_wiktionary': sample_new})

,new_entries_from_wiktionary
0,'fess up
1,ab off
2,abate of
3,absorb oneself in
4,abstract away from
5,abut on
6,account of
7,account to
8,accredit with
9,ace into


## Coverage checks for target expressions

In [5]:
targets = ['get going', 'screw with', 'get to know', 'plan ahead']
idiom_set = {e.get('phrase', '').strip().lower() for e in idioms if e.get('phrase')}
rows = []
for t in targets:
    rows.append({
        'expression': t,
        'in_idiom_lexicon': t in idiom_set,
        'in_base_pv_before': t in base_set,
        'in_wiktionary_pv_category': t in wiki_set,
        'in_final_pv_after_merge': t in merged_set,
        'in_any_current_lexicon': (t in idiom_set) or (t in merged_set),
    })
pd.DataFrame(rows)

,expression,in_idiom_lexicon,in_base_pv_before,in_wiktionary_pv_category,in_final_pv_after_merge,in_any_current_lexicon
0,get going,True,False,False,False,True
1,screw with,False,False,True,True,True
2,get to know,False,False,False,False,False
3,plan ahead,False,False,True,True,True


## Search helpers

In [6]:
final_df = pd.DataFrame({'canonical': sorted(merged_set)})


def search_pv(query: str, mode: str = 'contains', limit: int = 30) -> pd.DataFrame:
    q = query.strip().lower()
    s = final_df['canonical'].str.lower()
    if mode == 'exact':
        out = final_df[s == q]
    elif mode == 'prefix':
        out = final_df[s.str.startswith(q)]
    else:
        out = final_df[s.str.contains(q, regex=False)]
    return out.head(limit).reset_index(drop=True)


search_pv('screw with', mode='exact')

,canonical
0,screw with


In [7]:
search_pv('plan ahead', mode='exact')

,canonical
0,plan ahead


In [8]:
search_pv('get to know', mode='exact')

,canonical
